In [0]:
import sys
from pyspark.sql import functions as F

dbutils.widgets.removeAll()
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")

sys.path.append("/Workspace/Shared")
import aws_utils

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("run_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.landed.profits_customer_relation")
        .select(F.max("BusinessDate"))
        .first()[0]
    )


In [0]:


df_result = (
    spark.table(f"`{env}-cdh`.landed.profits_customer_relation")
    .filter(F.col("BusinessDate") == run_date)
    .select(
        F.col("FST_CUST_ID"),
        F.col("FST_CDI"),
        F.col("FST_NAME"),
        F.col("relation"),
        F.col("SEC_CUST_ID"),
        F.col("SEC_CDI"),
        F.col("SEC_NAME"),
        F.col("BusinessDate")
    )
)


In [0]:


(
    df_result.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{env}-cdh-delivery`.`it`.`CepalBILive_common_CustomerRelation`")
)
